# LoRA rank × base precision on a single T4: reproduction notebook

This notebook only calls the scripts in `scripts/`; all logic lives there. See `README.md` for the question, protocol and limitations.

**Runtime:** *Runtime → Change runtime type → T4 GPU.* The full grid (24 runs) takes roughly 2-3 GPU-hours; regenerating the figures from committed results takes seconds and needs no GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # set before running

import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt
# If Colab prints "You must restart the runtime", do Runtime → Restart session, then re-run the cell above (not this one).

## Path 1: regenerate every figure from the committed raw results (no GPU needed)

In [ ]:
!python scripts/make_figures.py

## Path 2: full re-run on your own GPU

Results go to `RESULTS_DIR`. A Colab session can drop mid-grid; `run_ablation.py` skips finished runs, so just re-run the cell. To survive a full disconnect, point `RESULTS_DIR` at Google Drive (uncomment the lines below).

In [ ]:
RESULTS_DIR = "results_rerun"  # reviewers: keep this. Author's reference run: "results" (the committed folder)
# from google.colab import drive; drive.mount("/content/drive")
# RESULTS_DIR = "/content/drive/MyDrive/lora-t4-ablation/results"
!python scripts/env_info.py --results-dir {RESULTS_DIR}

In [ ]:
# ~2 minute pipeline check on a tiny subset (writes to results_smoke/, never mixed with real results)
!python scripts/run_ablation.py --smoke --seed 0 --rank 8

In [ ]:
!python scripts/baseline_tfidf.py --results-dir {RESULTS_DIR}

In [ ]:
!python scripts/run_ablation.py --results-dir {RESULTS_DIR}

In [ ]:
!python scripts/audit.py --results-dir {RESULTS_DIR}
!python scripts/make_figures.py --results-dir {RESULTS_DIR} --figures-dir {RESULTS_DIR}/figures

## View results

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

out = Path(RESULTS_DIR) if Path(RESULTS_DIR, "summary.md").exists() else Path("results")
fig_dir = out / "figures" if (out / "figures").exists() else Path("figures")
display(Markdown(out.joinpath("summary.md").read_text()))
for png in sorted(fig_dir.glob("*.png")):
    display(Image(str(png)))

In [ ]:
# Optional: download the new results to commit them back to the repo
!zip -qr results_rerun.zip {RESULTS_DIR}
from google.colab import files; files.download("results_rerun.zip")